In [1]:
###############################################################################
# The Institute for the Design of Advanced Energy Systems Integrated Platform
# Framework (IDAES IP) was produced under the DOE Institute for the
# Design of Advanced Energy Systems (IDAES).
#
# Copyright (c) 2018-2026 by the software owners: The Regents of the
# University of California, through Lawrence Berkeley National Laboratory,
# National Technology & Engineering Solutions of Sandia, LLC, Carnegie Mellon
# University, West Virginia University Research Corporation, et al.
# All rights reserved.  Please see the files COPYRIGHT.md and LICENSE.md
# for full copyright and license information.
###############################################################################

# Parameter Estimation Using the NRTL State Block

Author: Jaffer Ghouse  
Maintainer: Stephen Cini  
Updated: 2026-06-11  

In this module, we use Pyomo's `parmest` tool in conjunction with IDAES models for parameter estimation. We demonstrate these tools by estimating the parameters associated with the NRTL property model for a benzene-toluene mixture. The NRTL model has 2 sets of parameters: the non-randomness parameter (`alpha_ij`) and the binary interaction parameter (`tau_ij`), where `i` and `j` are the pure component species. In this example, we only estimate the binary interaction parameter (`tau_ij`) for a given dataset. When estimating parameters associated with the property package, IDAES provides the flexibility of doing the parameter estimation by just using the state block or by using a unit model with a specified property package. This module will demonstrate parameter estimation by using only the state block. 

We will complete the following tasks:
* Set up a method to return an initialized model
* Set up the parameter estimation problem using `parmest`
* Analyze the results
* Demonstrate advanced features using `parmest`

## Key links to documentation:
* NRTL Model - https://idaes-pse.readthedocs.io/en/stable/reference_guides/model_libraries/generic/property_models/activity_coefficient.html
* parmest - https://pyomo.readthedocs.io/en/stable/explanation/analysis/parmest/index.html


<div class="alert alert-block alert-info">
<b>Inline Exercise:</b>
import `ConcreteModel`, `value`, and `Suffix` from Pyomo, `FlowsheetBlock` and `Flash` from IDAES. 
</div>

In [3]:
# Todo: import ConcreteModel, value, and Suffix from pyomo.environ
from pyomo.environ import ConcreteModel, value, Suffix

# Todo: import FlowsheetBlock from idaes.core
from idaes.core import FlowsheetBlock

# Todo: import Flash unit model from idaes.models.unit_models
from idaes.models.unit_models import Flash

In the next cell, we import the parameter block used in this module and the idaes logger. 

In [4]:
from idaes.models.properties.activity_coeff_models.BTX_activity_coeff_VLE import (
    BTXParameterBlock,
)
import idaes.logger as idaeslog

In the next cell, we import `parmest` from Pyomo and the `pandas` package. We need `pandas` as `parmest` uses `pandas.dataframe` for handling the input data and the results.

In [5]:
import pyomo.contrib.parmest.parmest as parmest
import pandas as pd

## Setting up an initialized model

We need to provide a method that returns an initialized model to the `parmest` tool in Pyomo.

<div class="alert alert-block alert-info">
<b>Inline Exercise:</b>
Using what you have learned from previous modules, fill in the missing code below to return an initialized IDAES model. 
</div>

In [7]:
def NRTL_model(data):

    # Todo: Create a ConcreteModel object
    m = ConcreteModel()

    # Todo: Create FlowsheetBlock object
    m.fs = FlowsheetBlock(dynamic=False)

    # Todo: Create a properties parameter object with the following options:
    # "valid_phase": ('Liq', 'Vap')
    # "activity_coeff_model": 'NRTL'
    m.fs.properties = BTXParameterBlock(
        valid_phase=("Liq", "Vap"), activity_coeff_model="NRTL"
    )
    m.fs.state_block = m.fs.properties.build_state_block(defined_state=True)

    # Fix the state variables on the state block
    # hint: state variables exist on the state block i.e. on m.fs.state_block

    m.fs.state_block.flow_mol.fix(1)
    m.fs.state_block.temperature.fix(368)
    m.fs.state_block.pressure.fix(101325)
    m.fs.state_block.mole_frac_comp["benzene"].fix(0.5)
    m.fs.state_block.mole_frac_comp["toluene"].fix(0.5)

    # Fix NRTL specific parameters.

    # non-randomness parameter - alpha_ij (set at 0.3, 0 if i=j)
    m.fs.properties.alpha["benzene", "benzene"].fix(0)
    m.fs.properties.alpha["benzene", "toluene"].fix(0.3)
    m.fs.properties.alpha["toluene", "toluene"].fix(0)
    m.fs.properties.alpha["toluene", "benzene"].fix(0.3)

    # binary interaction parameter - tau_ij (0 if i=j, else to be estimated later but fixing to initialize)
    m.fs.properties.tau["benzene", "benzene"].fix(0)
    m.fs.properties.tau["benzene", "toluene"].fix(-0.9)
    m.fs.properties.tau["toluene", "toluene"].fix(0)
    m.fs.properties.tau["toluene", "benzene"].fix(1.4)

    # Initialize the flash unit
    m.fs.state_block.initialize(outlvl=idaeslog.INFO_LOW)

    # Fix at actual temperature
    if isinstance(data, dict) or isinstance(data, pd.Series):
        m.fs.state_block.temperature.fix(float(data["temperature"]))
    elif isinstance(data, pd.DataFrame):
        m.fs.state_block.temperature.fix(float(data.iloc[0]["temperature"]))
    else:
        raise ValueError("Unrecognized data type.")

    # Set bounds on variables to be estimated
    m.fs.properties.tau["benzene", "toluene"].setlb(-5)
    m.fs.properties.tau["benzene", "toluene"].setub(5)

    m.fs.properties.tau["toluene", "benzene"].setlb(-5)
    m.fs.properties.tau["toluene", "benzene"].setub(5)

    # Return initialized flash model
    return m

In [8]:
from idaes.core.util.model_statistics import degrees_of_freedom
import pytest

# Testing the initialized model
test_data = {"temperature": 368}

m = NRTL_model(test_data)

# Check that degrees of freedom is 0
assert degrees_of_freedom(m) == 0

# Check for output values
assert value(m.fs.state_block.mole_frac_phase_comp["Liq", "benzene"]) == pytest.approx(
    0.389, abs=1e-2
)
assert value(m.fs.state_block.mole_frac_phase_comp["Vap", "benzene"]) == pytest.approx(
    0.610, abs=1e-2
)

assert value(m.fs.state_block.mole_frac_phase_comp["Liq", "toluene"]) == pytest.approx(
    0.610, abs=1e-2
)
assert value(m.fs.state_block.mole_frac_phase_comp["Vap", "toluene"]) == pytest.approx(
    0.394, abs=1e-2
)

## Parameter estimation using parmest

In addition to providing a method to return an initialized model, the `parmest` tool needs the following:

* Experiment class to set up and label model with suffixes
* Dataset with multiple scenarios - organized into an experiment list


Here we build an experiment class to label our model problem for parameter estimation. The labels are defined as a `Suffix`, and the main labels for our model are `experiment_outputs`, `unknown_parameters`. and `measurement_error`.

For this problem, the error will be computed for the mole fraction of benzene in the vapor and liquid phase between the model prediction and data. The `experimental_outputs` will therefore be the mole fraction of benzene in the two phases.

In this example, we only estimate the binary interaction parameter (`tau_ij`). Given that this variable is usually indexed as `tau_ij = Var(component_list, component_list)`, there are 2*2=4 degrees of freedom. However, when i=j, the binary interaction parameter is 0. Therefore, in this problem, we estimate the binary interaction parameter for the following variables only:

* fs.properties.tau['benzene', 'toluene']
* fs.properties.tau['toluene', 'benzene']

As shown below, these model components are used as our `unknown_parameters`.

We define `measurement_error` as none so parmest calculates the value internally based on the experimental outputs. Refer to https://pyomo.readthedocs.io/en/stable/explanation/analysis/parmest/driver.html for more information. 

In [9]:
# Build an experiment class to take advantage of new parmest interface
from pyomo.contrib.parmest.experiment import Experiment


class NRTLExperiment(Experiment):
    """Experiment class for parameter estimation of NRTL model using parmest"""

    def __init__(self, data, meas_error=None):
        """Initialize the NRTLExperiment class

        Args:
            data: DataFrame containing the experimental data
            meas_error: Measurement error for the data (optional)
        """
        self.model = None
        self.data = data
        self.meas_error = meas_error

    def create_model(self):
        """Create the Pyomo model for the NRTL parameter estimation problem"""
        self.model = NRTL_model(self.data)

    def label_model(self):
        import pyomo.environ as pyo
        import pandas as pd

        m = self.model

        # Parmest expects the first index of experiment outputs to be the data point
        # This is a workaround that will be addressed and corrected in a future release.

        m.data_point = pyo.Set(initialize=[0])

        # Wrap IDAES variables in Expressions indexed by data point
        m.liq_benzene_out = pyo.Expression(
            m.data_point,
            rule=lambda m, i: m.fs.state_block.mole_frac_phase_comp["Liq", "benzene"],
        )

        m.vap_benzene_out = pyo.Expression(
            m.data_point,
            rule=lambda m, i: m.fs.state_block.mole_frac_phase_comp["Vap", "benzene"],
        )

        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)

        m.experiment_outputs[m.liq_benzene_out[0]] = float(self.data["liq_benzene"])
        m.experiment_outputs[m.vap_benzene_out[0]] = float(self.data["vap_benzene"])

        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.measurement_error.update(
            [
                (m.liq_benzene_out[0], self.meas_error),
                (m.vap_benzene_out[0], self.meas_error),
            ]
        )

        # Add unknown parameters to the model for easier access
        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        m.unknown_parameters.update(
            (k, value(k))
            for k in [
                m.fs.properties.tau["benzene", "toluene"],
                m.fs.properties.tau["toluene", "benzene"],
            ]
        )

    def get_labeled_model(self):
        """Return the labeled model"""
        if self.model is None:
            self.create_model()
            self.label_model()
        return self.model

Pyomo's `parmest` tool supports the following data formats:
- pandas dataframe
- list of dictionaries
- list of json file names.

Please see the documentation for more details. 

For this example, we load data from the csv file `BT_NRTL_dataset.csv`. The dataset consists of fifty data points which provide the mole fraction of benzene in the vapor and liquid phase as a function of temperature. 

In [10]:
# Load data from csv
data = pd.read_csv("BT_NRTL_dataset.csv")

# Display the dataset
display(data)

,temperature,liq_benzene,vap_benzene
0,365.500000,0.480953,0.692110
1,365.617647,0.462444,0.667699
2,365.735294,0.477984,0.692441
3,365.852941,0.440547,0.640336
4,365.970588,0.427421,0.623328
5,366.088235,0.442725,0.647796
6,366.205882,0.434374,0.637691
7,366.323529,0.444642,0.654933
8,366.441176,0.427132,0.631229
9,366.558824,0.446301,0.661743


We define the `exp_list` by splitting the data into individual experiments, or data points.

In [11]:
# Update to new interface
exp_list = []
for i in range(data.shape[0]):
    exp_list.append(NRTLExperiment(data.iloc[i]))

We are now ready to set up the parameter estimation problem. We will create a parameter estimation object called pest. As shown below, we pass the experiment list, and an objective function to the Estimator method. tee=True will print the solver output after solving the parameter estimation problem.

In [12]:
import logging

idaeslog.getIdaesLogger("core.property_meta").setLevel(logging.ERROR)

pest = parmest.Estimator(exp_list, obj_function="SSE", tee=True)
obj_value, parameters = pest.theta_est()

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

In [13]:
# Check for values of the parameter estimation problem
assert obj_value == pytest.approx(5.07496e-4, 1e-3)
assert parameters["fs.properties.tau[benzene,toluene]"] == pytest.approx(-0.89876, 1e-3)
assert parameters["fs.properties.tau[toluene,benzene]"] == pytest.approx(1.41048, 1e-3)

You will notice that the resulting parameter estimation problem will have 1102 variables and 1100 constraints. Let us display the results by running the next cell. 

In [14]:
print("The SSE at the optimal solution is %0.6f" % (obj_value))
print()
print("The values for the parameters are as follows:")
for k, v in parameters.items():
    print(k, "=", v)

The SSE at the optimal solution is 0.000507

The values for the parameters are as follows:
fs.properties.tau[benzene,toluene] = -0.8987550041842163
fs.properties.tau[toluene,benzene] = 1.4104702103547941


Using the data that was provided, we have estimated the binary interaction parameters in the NRTL model for a benzene-toluene mixture. Although the dataset that was provided was temperature dependent, in this example we have estimated a single value that fits best for all temperatures.

### Advanced options for parmest: bootstrapping



Pyomo's `parmest` tool allows for bootstrapping where the parameter estimation is repeated over `n` samples with resampling from the original data set. Parameter estimation with bootstrap resampling can be used to identify confidence regions around each parameter estimate.  This analysis can be slow given the increased number of model instances that need to be solved. Please refer to https://pyomo.readthedocs.io/en/stable/contributed_packages/parmest/driver.html for more details. 

For the example above, the bootstrapping can be run by uncommenting the code in the following cell:

In [15]:
# Uncomment the following lines
# bootstrap_theta = pest.theta_est_bootstrap(4, seed=542)
# display(bootstrap_theta)